# GenAI-Assisted COBOL to Java Modernization (PoC)

## Objective

To explore and demonstrate the capabilities of generative AI in modernizing legacy systems by building a proof-of-concept that:

- Translates COBOL programs into clean, object-oriented Java code.
- Automatically generates technical documentation for the translated code.
- Evaluates the quality of the translation using structured prompts and large language models (LLMs).

## Import Packages

In [1]:
# !pip uninstall -y transformers sentence-transformers accelerate tokenizers huggingface_hub

In [2]:
# # Install compatible stack
# !pip install transformers==4.41.2
# !pip install accelerate==0.29.3
# !pip install langchain==0.2.1
# !pip install langchain-huggingface==0.0.3
# !pip install gradio==4.38.1
# !pip install sacrebleu==2.4.2

In [3]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
import sacrebleu
import logging


In [4]:
## Configure Logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("cobol_to_java_poc")


## Load Transformer LLM

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_name = "Salesforce/codegen-350M-mono"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.2,
    truncation=False
)

llm = HuggingFacePipeline(pipeline=generator)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## Prompt Templates

In [6]:
## translation
translation_prompt = PromptTemplate.from_template("""
You are a senior software engineer modernizing legacy COBOL systems.

Task: Convert the following COBOL program into a clean, object-oriented Java class.
Rules:
- Map COBOL variables to Java fields with proper types.
- Use constructors and methods to implement PROCEDURE DIVISION logic.
- Use System.out.println() for DISPLAY statements.
- Do NOT include any COBOL syntax in your output.
- Produce only valid Java code.
- Do not repeat imports, Do not include duplicate or incomplete argument definitions.
- Produce clean, readable, fully functional code.
- Stop outputting after the last line of the program.

COBOL:
{cobol_code}

Java:
""")

## Documentation
doc_prompt = PromptTemplate.from_template("""
Generate Javadoc-style documentation for the following Java code:
- Include class description
- Include field descriptions
- Include method descriptions and example usage

Java Code:
{java_code}

Documentation:
""")

### Create LangChain LLMChains

In [7]:
translation_chain = translation_prompt | llm # Converts COBOL to Java
doc_chain = doc_prompt | llm  # generates technical documentation for Java code

## Evaluation Function

In [8]:
def evaluate_translation(references, predictions):
    if not references or not references[0].strip():
        return 0.0
    bleu = sacrebleu.corpus_bleu(predictions, [references])
    return bleu.score

## Function for Translation, Documentation, Evaluation

In [9]:
# This function handles the complete pipeline:
# 1. Translates COBOL code to Java
# 2. Evaluates the translation using BLEU score
# 3. Generates technical documentation

def full_pipeline(cobol_code, reference_java):

    if not cobol_code.strip():
        return "No COBOL code provided", "", ""

    warning = ""
    if len(cobol_code) > 1500:
        warning = "Input is long and may be truncated by the model.\n"

    logger.info("Starting COBOL → Java translation pipeline")

    try:
        # Translate COBOL → Java
        java_code = translation_chain.invoke(
            {"cobol_code": cobol_code}
        )

        logger.info("Translation completed")

        # BLEU Evaluation
        bleu_score = evaluate_translation(
            [reference_java],
            [java_code]
        )

        logger.info(f"BLEU Score: {bleu_score:.2f}")

        # Documentation generation
        documentation = doc_chain.invoke(
            {"java_code": java_code}
        )

        logger.info("Documentation generation completed")

    except Exception as e:
        logger.error(f"Error in pipeline: {e}")
        return f"Pipeline error: {e}", "", ""

    print(java_code)
    print("\n" + "-"*80 + "\n")
    print(documentation)
    print("\n" + "-"*80 + "\n")
    print(f"BLEU Score: {bleu_score:.2f}")


In [10]:
default_cobol = """IDENTIFICATION DIVISION.
       PROGRAM-ID. PAYROLL.

       DATA DIVISION.
       WORKING-STORAGE SECTION.
       01 EMP-HOURS  PIC 9(3)V99 VALUE 40.00.
       01 EMP-RATE   PIC 9(3)V99 VALUE 25.50.
       01 EMP-PAY    PIC 9(5)V99.

       PROCEDURE DIVISION.
           COMPUTE EMP-PAY = EMP-HOURS * EMP-RATE
           DISPLAY 'Pay: ' EMP-PAY
           STOP RUN.

"""

default_reference = """public class Payroll {

    private double hours = 40.0;
    private double rate = 25.50;
    private double pay;

    public Payroll() {
        this.pay = hours * rate;
    }

    public void displayPay() {
        System.out.println("Pay: " + pay);
    }

    public static void main(String[] args) {
        Payroll payroll = new Payroll();
        payroll.displayPay();
    }
}
"""


In [11]:
full_pipeline(default_cobol,default_reference)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



You are a senior software engineer modernizing legacy COBOL systems.

Task: Convert the following COBOL program into a clean, object-oriented Java class.
Rules:
- Map COBOL variables to Java fields with proper types.
- Use constructors and methods to implement PROCEDURE DIVISION logic.
- Use System.out.println() for DISPLAY statements.
- Do NOT include any COBOL syntax in your output.
- Produce only valid Java code.
- Do not repeat imports, Do not include duplicate or incomplete argument definitions.
- Produce clean, readable, fully functional code.
- Stop outputting after the last line of the program.

COBOL:
             IDENTIFICATION DIVISION.
       PROGRAM-ID. PAYROLL.

       DATA DIVISION.
       WORKING-STORAGE SECTION.
       01 EMP-HOURS  PIC 9(3)V99 VALUE 40.00.
       01 EMP-RATE   PIC 9(3)V99 VALUE 25.50.
       01 EMP-PAY    PIC 9(5)V99.

       PROCEDURE DIVISION.
           COMPUTE EMP-PAY = EMP-HOURS * EMP-RATE
           DISPLAY 'Pay: ' EMP-PAY
           STOP RUN.


## Conclusion

- This PoC shows that generative AI can assist in modernizing legacy COBOL systems by translating them into clean, object-oriented Java code.
- The pipeline produces functional Java classes, generates documentation, and evaluates translation quality with BLEU scores.  
- While effective for simple programs, further enhancements are needed for complex COBOL constructs and large-scale systems. Overall, it demonstrates AI’s potential to accelerate legacy modernization and reduce manual effort.